# Atividade Prática — Modelos de Linguagem N-Grama do Zero

**Disciplina:** Modelos de Linguagem de Grande Escala — UFCG
**Prof.:** Leandro B. Marinho

## Objetivo
Implementar **do zero** (sem NLTK, spaCy, KenLM etc.) modelos de linguagem **unigrama, bigrama e trigrama**, treiná-los em um livro do **Projeto Gutenberg** e comparar os modelos pela **perplexidade** no conjunto de teste.

## Regras
- Só é permitido usar a biblioteca padrão do Python (`re`, `math`, `random`, `collections`, `urllib`) e `matplotlib` para os gráficos.
- **Vocês podem usar IA (ChatGPT, Claude, Copilot etc.) para gerar e ajustar o código.** Em compensação:
  1. todas as células de **verificação** (`assert`) precisam passar;
  2. vocês devem conseguir **explicar qualquer trecho** do código entregue;
  3. preencham a seção **"Declaração de uso de IA"** no final do notebook.
- Os trechos marcados com `# TODO` precisam ser implementados. Não altere as células de verificação.

## Roteiro
1. Carregar e limpar um livro do Gutenberg
2. Segmentar em sentenças e tokenizar
3. Separar treino / validação / teste e tratar palavras fora do vocabulário (`<UNK>`)
4. Implementar a classe `NGramLM` (contagens + MLE + suavização add-k)
5. Calcular a perplexidade de unigrama, bigrama e trigrama
6. Ajustar `k` na validação e implementar interpolação linear
7. Gerar texto no estilo Shannon
8. Responder às questões de análise


## 0. Imports e configuração

In [ ]:
# Bibliotecas permitidas pela atividade.
# Elas cobrem leitura de arquivos, expressões regulares, matemática,
# sorteios reprodutíveis, download e contagem de frequências.
import os
import re
import math
import random
import urllib.request
from collections import Counter

# Matplotlib será usado somente para os gráficos da atividade.
import matplotlib.pyplot as plt

# A semente torna a divisão dos dados e a geração de texto reproduzíveis.
SEED = 42
random.seed(SEED)

print("Ambiente carregado. Execute as células na ordem usando 'Run All'.")

## 1. Carregando um livro do Projeto Gutenberg

Sugestão: **Dom Casmurro**, de Machado de Assis (ID `55752`). Vocês podem escolher outro livro em português (ou em outro idioma) — basta trocar o ID em https://www.gutenberg.org. Prefiram livros longos (pelo menos ~50 mil palavras).

Os arquivos do Gutenberg têm um **cabeçalho** e um **rodapé** de licença que precisam ser removidos. Eles ficam antes da linha `*** START OF THE PROJECT GUTENBERG EBOOK ... ***` e depois da linha `*** END OF THE PROJECT GUTENBERG EBOOK ... ***`.

> Se não houver internet, baixe o `.txt` manualmente e coloque-o na mesma pasta do notebook com o nome `livro.txt`.

In [ ]:
LIVRO_ID = 55752  # Dom Casmurro
URL = f"https://www.gutenberg.org/cache/epub/{LIVRO_ID}/pg{LIVRO_ID}.txt"
CAMINHO_LOCAL = "livro.txt"


def carregar_livro(url, caminho_local=CAMINHO_LOCAL):
    """Lê o livro localmente ou baixa uma cópia do Projeto Gutenberg."""
    if os.path.exists(caminho_local):
        # Ler em UTF-8 preserva os acentos do texto em português.
        with open(caminho_local, "r", encoding="utf-8") as arquivo:
            return arquivo.read()

    # O arquivo é baixado somente na primeira execução.
    with urllib.request.urlopen(url, timeout=30) as resposta:
        texto = resposta.read().decode("utf-8")

    # Guardar uma cópia evita depender da internet nas próximas execuções.
    with open(caminho_local, "w", encoding="utf-8") as arquivo:
        arquivo.write(texto)
    return texto


def remover_cabecalho_rodape(texto):
    """Retorna somente o conteúdo entre os marcadores do Gutenberg."""
    inicio = re.search(r"^\s*\*\*\* START OF .*?\*\*\*\s*$", texto, flags=re.IGNORECASE | re.MULTILINE)
    fim = re.search(r"^\s*\*\*\* END OF .*?\*\*\*\s*$", texto, flags=re.IGNORECASE | re.MULTILINE)

    # Se os marcadores não existirem, mantemos o texto para não apagá-lo.
    if inicio is None or fim is None or fim.start() <= inicio.end():
        return texto.strip()
    return texto[inicio.end():fim.start()].strip()


texto_bruto = carregar_livro(URL)
texto = remover_cabecalho_rodape(texto_bruto)
print(f"Caracteres (bruto): {len(texto_bruto):,} | após limpeza: {len(texto):,}")
print(texto[:500])

In [ ]:
# Verificação — não altere
assert "START OF" not in texto.upper()[:2000], "O cabeçalho do Gutenberg não foi removido"
assert "END OF THE PROJECT GUTENBERG" not in texto.upper(), "O rodapé do Gutenberg não foi removido"
assert len(texto) > 100_000, "Texto muito curto: escolha um livro maior"
print("OK")

## 2. Segmentação em sentenças e tokenização

Implemente:
- `segmentar_sentencas(texto)`: divide o texto em sentenças (uma regra simples com `re.split` em `.`, `!`, `?` e `…` já basta; comente as limitações nas questões).
- `tokenizar(sentenca)`: converte para minúsculas e devolve a lista de palavras. Decidam e **justifiquem** se vão manter a pontuação como token. A solução de referência mantém só palavras (inclusive acentuadas e com hífen, como *disse-lhe*).

Descarte sentenças vazias.

In [ ]:
def segmentar_sentencas(texto):
    """Divide o texto em sentenças usando sinais de pontuação finais."""
    # Esta é uma regra simples: cada ponto, exclamação, interrogação
    # ou reticência é tratado como possível fim de sentença.
    partes = re.split(r"[.!?…]+", texto)

    # Remover espaços nas extremidades e descartar partes vazias.
    return [parte.strip() for parte in partes if parte.strip()]


def tokenizar(sentenca):
    """Converte uma sentença em palavras minúsculas, sem pontuação."""
    sentenca = sentenca.lower()

    # A expressão aceita letras ASCII, letras acentuadas e palavras com hífen.
    # Assim, por exemplo, "disse-lhe" continua sendo um único token.
    padrao_palavra = r"[a-zà-öø-ÿ]+(?:-[a-zà-öø-ÿ]+)*"
    return re.findall(padrao_palavra, sentenca)


# Cada sentença é transformada em uma lista de tokens.
sentencas = [tokenizar(s) for s in segmentar_sentencas(texto)]
sentencas = [s for s in sentencas if len(s) > 0]

n_tokens = sum(len(s) for s in sentencas)
print(f"Sentenças: {len(sentencas):,} | tokens: {n_tokens:,} | tipos: {len(set(w for s in sentencas for w in s)):,}")
print(sentencas[:3])

In [ ]:
# Verificação — não altere
assert tokenizar("Ela disse-lhe: Não!")[0] == "ela", "Converta para minúsculas"
assert len(segmentar_sentencas("Oi. Tudo bem? Sim!")) == 3
assert all(isinstance(s, list) and len(s) > 0 for s in sentencas)
print("OK")

### 2.1 Lei de Zipf (rápido)
Plote, em escala log-log, a frequência das palavras em função do rank. O gráfico se aproxima de uma reta? O que acontece nas pontas?

In [ ]:
freq = Counter(w for s in sentencas for w in s)

# Ordenar as frequências da maior para a menor cria os ranks de Zipf.
frequencias_ordenadas = sorted(freq.values(), reverse=True)
ranks = range(1, len(frequencias_ordenadas) + 1)

# A escala log-log facilita observar a relação aproximada entre rank e frequência.
plt.figure(figsize=(8, 5))
plt.loglog(ranks, frequencias_ordenadas, marker=".", linestyle="none", markersize=3)
plt.title("Lei de Zipf no corpus")
plt.xlabel("Rank da palavra")
plt.ylabel("Frequência")
plt.grid(True, which="both", alpha=0.25)
plt.show()

print(f"Palavra mais frequente: {freq.most_common(1)[0]}")
print("Nas palavras muito frequentes e muito raras, a curva tende a se afastar da reta ideal de Zipf.")

## 3. Treino, validação e teste + tratamento de `<UNK>`

1. Embaralhe as sentenças com `SEED` e separe **80% treino / 10% validação / 10% teste**.
2. Construa o vocabulário **apenas com o treino**. Palavras que aparecem **menos de `MIN_FREQ` vezes** no treino viram `<UNK>`.
3. Substitua por `<UNK>` as palavras de validação e teste que não estão no vocabulário.

> Por que isso é necessário? Sem `<UNK>`, qualquer palavra nova no teste tem probabilidade zero, e a perplexidade vai para infinito (slide *Zeros*).

In [ ]:
MIN_FREQ = 2
UNK = "<UNK>"


def dividir(sentencas, prop_treino=0.8, prop_val=0.1, seed=SEED):
    """Embaralha as sentenças e separa treino, validação e teste."""
    if not 0 < prop_treino < 1 or not 0 <= prop_val < 1:
        raise ValueError("As proporções devem estar entre 0 e 1.")
    if prop_treino + prop_val >= 1:
        raise ValueError("Treino e validação devem deixar dados para o teste.")

    # Usar uma cópia evita alterar a ordem da lista original.
    embaralhadas = list(sentencas)
    random.Random(seed).shuffle(embaralhadas)

    total = len(embaralhadas)
    fim_treino = int(total * prop_treino)
    fim_val = int(total * (prop_treino + prop_val))
    return (
        embaralhadas[:fim_treino],
        embaralhadas[fim_treino:fim_val],
        embaralhadas[fim_val:]
    )


def construir_vocab(treino, min_freq=MIN_FREQ):
    """Cria o vocabulário apenas com palavras frequentes do treino."""
    contagens = Counter(w for sentenca in treino for w in sentenca)

    # Palavras raras não entram no vocabulário e serão convertidas em <UNK>.
    return {w for w, contagem in contagens.items() if contagem >= min_freq}


def substituir_unk(sentencas, vocab):
    """Troca cada palavra ausente do vocabulário por <UNK>."""
    return [
        [w if w in vocab else UNK for w in sentenca]
        for sentenca in sentencas
    ]


# A divisão acontece antes da construção do vocabulário para evitar vazamento.
treino_raw, val_raw, teste_raw = dividir(sentencas)
vocab = construir_vocab(treino_raw)
treino = substituir_unk(treino_raw, vocab)
val = substituir_unk(val_raw, vocab)
teste = substituir_unk(teste_raw, vocab)

taxa_unk = sum(w == UNK for s in teste for w in s) / sum(len(s) for s in teste)
print(f"Treino: {len(treino):,} | Val: {len(val):,} | Teste: {len(teste):,} sentenças")
print(f"Vocabulário: {len(vocab):,} palavras (+ <UNK>) | taxa de <UNK> no teste: {taxa_unk:.2%}")

In [ ]:
# Verificação — não altere
assert len(treino) + len(val) + len(teste) == len(sentencas)
assert abs(len(treino) / len(sentencas) - 0.8) < 0.01
assert all(w in vocab or w == UNK for s in teste for w in s)
print("OK")

## 4. A classe `NGramLM`

Implemente um modelo n-grama genérico (funciona para `n = 1, 2, 3, ...`):

- **Padding:** cada sentença vira `['<s>'] * (n-1) + sentença + ['</s>']`. Assim, no trigrama, a primeira palavra é condicionada em `(<s>, <s>)`.
- **Contagens:** `C(contexto, w)` e `C(contexto)`, em que o contexto é a tupla das `n-1` palavras anteriores (para o unigrama, o contexto é a tupla vazia `()`).
- **Vocabulário do modelo** `V`: palavras do treino + `<UNK>` + `</s>` (o `<s>` **não** entra, porque nunca é previsto).
- **Probabilidade com suavização add-k** (com `k = 0` você obtém a MLE; com `k = 1`, Laplace):

$$P(w \mid \text{ctx}) = \frac{C(\text{ctx}, w) + k}{C(\text{ctx}) + k\,|V|}$$

- **Log-probabilidade de uma sentença**: soma dos `log P` de cada palavra **e do `</s>`**.
- **Perplexidade** de um conjunto de sentenças, com $N$ = total de palavras previstas (incluindo os `</s>`):

$$PP = \exp\left(-\frac{1}{N}\sum \log P(w_i \mid \text{ctx}_i)\right)$$

Se alguma probabilidade for zero, a perplexidade deve ser `float('inf')`.

In [ ]:
BOS, EOS = "<s>", "</s>"


class NGramLM:
    """Modelo de linguagem n-grama com MLE e suavização add-k."""

    def __init__(self, n, k=0.0):
        if n < 1:
            raise ValueError("A ordem n deve ser pelo menos 1.")
        if k < 0:
            raise ValueError("O valor de k não pode ser negativo.")

        self.n = n
        self.k = k
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.vocab = set()

    def _padding(self, sentenca):
        """Adiciona marcadores de início e fim à sentença."""
        return [BOS] * (self.n - 1) + list(sentenca) + [EOS]

    def _ngramas(self, sentenca):
        """Gera pares (contexto, palavra) de uma sentença já preenchida."""
        pares = []
        for indice in range(len(sentenca) - self.n + 1):
            # Para n=1, o contexto é a tupla vazia.
            contexto = tuple(sentenca[indice:indice + self.n - 1])
            palavra = sentenca[indice + self.n - 1]
            pares.append((contexto, palavra))
        return pares

    def fit(self, sentencas):
        """Conta n-gramas e contextos observados no corpus de treino."""
        # Limpar as contagens permite reutilizar o mesmo objeto com outro corpus.
        self.ngram_counts.clear()
        self.context_counts.clear()
        self.vocab.clear()

        for sentenca in sentencas:
            self.vocab.update(sentenca)
            padded = self._padding(sentenca)

            for contexto, palavra in self._ngramas(padded):
                self.ngram_counts[(contexto, palavra)] += 1
                self.context_counts[contexto] += 1

        # UNK pode aparecer no teste e EOS é sempre uma palavra prevista.
        self.vocab.update({UNK, EOS})
        return self

    @property
    def V(self):
        """Tamanho do vocabulário usado no denominador da probabilidade."""
        return len(self.vocab)

    def prob(self, w, ctx):
        """Calcula P(w | ctx) usando MLE ou suavização add-k."""
        contexto = tuple(ctx)
        if w not in self.vocab:
            return 0.0

        contagem_contexto = self.context_counts[contexto]
        contagem_ngram = self.ngram_counts[(contexto, w)]
        denominador = contagem_contexto + self.k * self.V

        # Com k=0, contexto nunca visto tem denominador igual a zero.
        if denominador == 0:
            return 0.0
        return (contagem_ngram + self.k) / denominador

    def logprob_sentenca(self, sentenca):
        """Soma os logaritmos das probabilidades, incluindo EOS."""
        logprob = 0.0
        for contexto, palavra in self._ngramas(self._padding(sentenca)):
            probabilidade = self.prob(palavra, contexto)
            if probabilidade <= 0:
                return float("-inf")
            logprob += math.log(probabilidade)
        return logprob

    def perplexidade(self, sentencas):
        """Calcula exp(-média dos log-probabilidades do corpus)."""
        soma_logprob = 0.0
        total_previsto = 0

        for sentenca in sentencas:
            logprob = self.logprob_sentenca(sentenca)
            if logprob == float("-inf"):
                return float("inf")
            soma_logprob += logprob
            total_previsto += len(sentenca) + 1  # +1 corresponde ao EOS.

        if total_previsto == 0:
            return float("inf")
        return math.exp(-soma_logprob / total_previsto)

### 4.1 Verificação com o corpus dos slides

Use o mini-corpus da aula (*I am Sam*). Os valores esperados são os mesmos calculados em sala.

In [ ]:
# Verificação — não altere
corpus_sam = [["I", "am", "Sam"], ["Sam", "I", "am"], ["I", "do", "not", "like", "green", "eggs", "and", "ham"]]
bg = NGramLM(2, k=0).fit(corpus_sam)

assert math.isclose(bg.prob("I", ["<s>"]), 2/3)
assert math.isclose(bg.prob("Sam", ["<s>"]), 1/3)
assert math.isclose(bg.prob("am", ["I"]), 2/3)
assert math.isclose(bg.prob("</s>", ["Sam"]), 1/2)
assert math.isclose(bg.prob("Sam", ["am"]), 1/2)
assert math.isclose(bg.prob("do", ["I"]), 1/3)

# As probabilidades de um contexto devem somar 1 (MLE e add-k)
for k in [0, 1, 0.1]:
    m = NGramLM(2, k=k).fit(corpus_sam)
    assert math.isclose(sum(m.prob(w, ["I"]) for w in m.vocab), 1.0), f"não soma 1 com k={k}"
    m3 = NGramLM(3, k=k).fit(corpus_sam)
    assert math.isclose(sum(m3.prob(w, ["<s>", "I"]) for w in m3.vocab), 1.0)

# Unigrama: P(I) = 3/17 (14 palavras + 3 </s> = 17 tokens previstos; o <s> não conta)
ug = NGramLM(1, k=0).fit(corpus_sam)
assert math.isclose(ug.prob("I", []), 3 / 17)

# MLE atribui zero a bigrama não visto -> perplexidade infinita
assert NGramLM(2, k=0).fit(corpus_sam).perplexidade([["Sam", "do", "ham"]]) == float("inf")
print("OK")

## 5. Perplexidade: unigrama × bigrama × trigrama

### 5.1 MLE (sem suavização)
Treine os três modelos com `k = 0` e calcule a perplexidade no **treino** e no **teste**. Explique o que acontece.

In [ ]:
resultados_mle = {}
for n in [1, 2, 3]:
    # TODO: treine com k=0 e calcule PP no treino e no teste
    pass

### 5.2 Suavização de Laplace (add-1)
Repita com `k = 1`.

In [ ]:
resultados_laplace = {}
for n in [1, 2, 3]:
    # TODO: treine com k=1 e calcule PP no treino e no teste
    pass

### 5.3 Ajustando `k` na validação (add-k)
Para cada `n`, escolha o melhor `k` em `[1, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001]` usando a **validação**. Depois avalie **uma única vez** no teste.

In [ ]:
KS = [1, 0.5, 0.1, 0.05, 0.01, 0.005, 0.001]
melhor_k, resultados_addk = {}, {}
for n in [1, 2, 3]:
    # TODO
    pass

## 6. Interpolação linear

Combine os três modelos MLE:

$$\hat P(w_i \mid w_{i-2}, w_{i-1}) = \lambda_1 P(w_i) + \lambda_2 P(w_i \mid w_{i-1}) + \lambda_3 P(w_i \mid w_{i-2}, w_{i-1}), \quad \sum \lambda_j = 1$$

Escolha os λ por **busca em grade** na validação (passo de 0.1, com λ₁ > 0 para evitar zeros) e reporte a perplexidade no teste.

> Dica: para facilitar, a classe `Interpolado` pode reaproveitar `NGramLM.perplexidade`, sobrescrevendo apenas `n` e `prob`.

In [ ]:
class Interpolado(NGramLM):
    def __init__(self, modelos, lambdas):
        self.modelos, self.lambdas = modelos, lambdas
        self.n = max(m.n for m in modelos)

    def prob(self, w, ctx):
        # TODO
        raise NotImplementedError


mle = [NGramLM(n, k=0).fit(treino) for n in [1, 2, 3]]
# TODO: busca em grade dos lambdas na validação e avaliação no teste
melhores_lambdas, pp_interp = None, None

### 6.1 Tabela e gráfico comparativo

In [ ]:
# TODO: tabela e gráfico de barras com a perplexidade no teste de cada modelo


## 7. Gerando texto no estilo Shannon

Implemente `gerar(modelo, max_palavras=30)`: comece com o contexto `<s>` (ou `<s> <s>`), sorteie a próxima palavra **proporcionalmente às contagens MLE** do contexto atual e continue até sortear `</s>`. Gere 3 frases com cada modelo (unigrama, bigrama, trigrama) e compare.

In [ ]:
def gerar(modelo, max_palavras=30, rng=random.Random(SEED)):
    # TODO
    raise NotImplementedError


for n, m in zip([1, 2, 3], mle):
    print(f"--- {n}-grama ---")
    for _ in range(3):
        print("•", gerar(m))

## 8. Questões de análise

Responda em células de texto, **com base nos seus resultados**:

1. O que aconteceu com a perplexidade no teste dos modelos MLE (seção 5.1)? E no treino? Explique por que o unigrama MLE não teve o mesmo problema.
2. Com Laplace (k = 1), a ordem "trigrama < bigrama < unigrama" dos slides (962 / 170 / 109) se manteve? Se não, explique o motivo, relacionando com a **esparsidade** (quantos trigramas possíveis existem, dado o tamanho do vocabulário, e quantos foram observados?).
3. Como o melhor `k` variou com `n`? Por que faz sentido que modelos de ordem maior prefiram `k` menor?
4. Compare add-k e interpolação. Quais λ foram escolhidos e o que eles dizem sobre a confiabilidade de cada modelo neste corpus?
5. Por que os hiperparâmetros (`k`, λ) foram ajustados na **validação** e não no **teste**? O que aconteceria se usássemos o teste?
6. Compare as frases geradas pelos três modelos. O que muda de um para o outro? O trigrama está "copiando" o livro? Como verificar isso?
7. Cite duas limitações da segmentação/tokenização que você usou, com exemplos do próprio livro (ex.: *Sr.*, *D.*, travessões de diálogo).
8. **(Bônus)** Troque o livro por outro autor ou época e avalie o modelo treinado em *Dom Casmurro* no novo livro. O que acontece com a perplexidade? Relacione com o slide "O Wall Street Journal não é Shakespeare".


*Respostas:*

1.
2.
3.
4.
5.
6.
7.


## 9. Declaração de uso de IA

- Ferramenta(s) usada(s):
- Para quais partes:
- Exemplos de prompts relevantes:
- O que precisou ser corrigido ou ajustado no código gerado:
- Algo que a IA errou ou explicou de forma incorreta:


---
### Critérios de avaliação
| Item | Peso |
|---|---|
| Carregamento, limpeza e tokenização (seções 1–3) | 15% |
| Classe `NGramLM` correta (todas as verificações passando) | 25% |
| Perplexidade MLE / Laplace / add-k (seção 5) | 20% |
| Interpolação linear (seção 6) | 10% |
| Geração estilo Shannon (seção 7) | 5% |
| Questões de análise (seção 8) | 20% |
| Declaração de uso de IA | 5% |
